# Week 1: Data, classical baselines, and cross-fitted teacher

Project: *Distilling genome-wide classical models into small quantum classifiers for breast cancer subtyping*

This notebook does four things:

1. Downloads TCGA-BRCA RNA-seq and PAM50 subtype labels from UCSC Xena.
2. Removes the 50 PAM50 genes, so models cannot re-learn the label definition.
3. Trains classical baselines on (a) the genome-wide feature set and (b) PCA-compressed 4, 6 and 8 dimensional features, with every data-dependent step fitted inside the training fold.
4. Produces cross-fitted soft labels from the classical teacher, saved for the week 3 distillation experiments, plus the fold-wise compressed features for the week 2 quantum kernels.

Everything is saved under `outputs/week1/` so weeks 2 to 4 reuse the exact same splits.

## 0. Configuration

In [ ]:
import os, json, time, warnings, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     GridSearchCV, cross_val_predict)
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
DATA_DIR = Path("data")
OUT_DIR = Path("outputs/week1")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5          # outer CV folds
N_REPEATS = 3         # outer CV repeats (quantum weeks may use repeat 0 only)
INNER_SPLITS = 5      # inner folds for cross-fitted teacher soft labels
TOP_VAR_GENES = 5000  # genes kept by the in-fold variance filter
PCA_DIMS = [4, 6, 8]  # compressed dimensions = qubit counts for weeks 2 and 3
TEACHER_REPEATS = [0] # outer repeats for which teacher soft labels are produced
TEACHER_NAME = "LogReg_L2"  # change after looking at section 6 results if another model is clearly better

# Set True to run the whole notebook on synthetic data (no download), useful for testing.
DEMO_MODE = False

EXPR_URL = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FHiSeqV2.gz"
CLIN_URL = "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FBRCA_clinicalMatrix"
EXPR_PATH = DATA_DIR / "TCGA_BRCA_HiSeqV2.tsv.gz"
CLIN_PATH = DATA_DIR / "TCGA_BRCA_clinicalMatrix.tsv"

## 1. Download data

`HiSeqV2` is the gene-level RSEM expression already transformed as log2(x + 1), so we do **not** log-transform again.
If the download fails (for example behind a firewall), download both files manually from the UCSC Xena TCGA hub
(cohort: *TCGA Breast Cancer (BRCA)*) and place them at the paths in the config cell.

In [ ]:
def download(url, path):
    if path.exists():
        print(f"Found {path}")
        return
    print(f"Downloading {url}")
    t0 = time.time()
    urllib.request.urlretrieve(url, path)
    print(f"  saved {path} ({path.stat().st_size / 1e6:.1f} MB, {time.time() - t0:.0f}s)")

if not DEMO_MODE:
    download(EXPR_URL, EXPR_PATH)
    download(CLIN_URL, CLIN_PATH)

## 2. Load and align expression with PAM50 labels

In [ ]:
def make_demo_data(n=600, n_genes=2000, seed=SEED):
    # Synthetic stand-in with the same shapes and naming conventions as the real files.
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=n, n_features=n_genes, n_informative=60,
                               n_redundant=40, n_classes=5, n_clusters_per_class=1,
                               weights=[0.45, 0.2, 0.1, 0.18, 0.07], random_state=seed)
    genes = [f"GENE{i}" for i in range(n_genes)]
    genes[:6] = ["ESR1", "ERBB2", "MKI67", "KNTC2", "ORC6L", "CDCA1"]  # to test PAM50 removal
    samples = [f"TCGA-AA-{i:04d}-01" for i in range(n)]
    expr = pd.DataFrame(X, index=samples, columns=genes)
    names = np.array(["LumA", "LumB", "Her2", "Basal", "Normal"])
    labels = pd.Series(names[y], index=samples, name="PAM50")
    return expr, labels

if DEMO_MODE:
    expr, labels = make_demo_data()
else:
    raw = pd.read_csv(EXPR_PATH, sep="\t", index_col=0)      # genes x samples
    expr = raw.T                                                # samples x genes
    clin = pd.read_csv(CLIN_PATH, sep="\t", index_col=0)
    labels = clin["PAM50Call_RNAseq"].dropna()
    labels.name = "PAM50"

# Keep primary tumours only (sample type code 01) and one sample per patient.
expr = expr[[s[13:15] == "01" for s in expr.index]]
expr = expr[~pd.Index([s[:12] for s in expr.index]).duplicated()]

common = expr.index.intersection(labels.index)
expr, labels = expr.loc[common], labels.loc[common]
expr = expr.loc[:, expr.notna().all(axis=0)]   # drop genes with any missing value

print("Samples x genes:", expr.shape)
print(labels.value_counts())

## 3. Remove the PAM50 genes

PAM50 subtype calls are computed from these 50 genes. Keeping them would let the model partly re-learn the labelling rule.
Older gene symbols used in some TCGA files are included as aliases (NDC80 = KNTC2, ORC6 = ORC6L, NUF2 = CDCA1).

In [ ]:
PAM50_GENES = {
    "ACTR3B", "ANLN", "BAG1", "BCL2", "BIRC5", "BLVRA", "CCNB1", "CCNE1", "CDC20", "CDC6",
    "CDH3", "CENPF", "CEP55", "CXXC5", "EGFR", "ERBB2", "ESR1", "EXO1", "FGFR4", "FOXA1",
    "FOXC1", "GPR160", "GRB7", "KIF2C", "KRT14", "KRT17", "KRT5", "MAPT", "MDM2", "MELK",
    "MIA", "MKI67", "MLPH", "MMP11", "MYBL2", "MYC", "NAT1", "NDC80", "NUF2", "ORC6",
    "PGR", "PHGDH", "PTTG1", "RRM2", "SFRP1", "SLC39A6", "TMEM45B", "TYMS", "UBE2C", "UBE2T",
}
PAM50_ALIASES = {"KNTC2", "ORC6L", "CDCA1"}
to_drop = sorted((PAM50_GENES | PAM50_ALIASES) & set(expr.columns))
expr = expr.drop(columns=to_drop)
print(f"Removed {len(to_drop)} PAM50 genes/aliases: {to_drop}")
print("Remaining genes:", expr.shape[1])

## 4. Encode labels and fix the outer CV splits

In [ ]:
le = LabelEncoder()
y = le.fit_transform(labels.values)
CLASSES = list(le.classes_)
X = expr.values.astype(np.float32)
K = len(CLASSES)
print("Classes:", dict(zip(CLASSES, np.bincount(y))))

rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
splits = []
for i, (tr, te) in enumerate(rskf.split(X, y)):
    splits.append({"repeat": i // N_SPLITS, "fold": i % N_SPLITS,
                   "train": tr.tolist(), "test": te.tolist()})

with open(OUT_DIR / "splits.json", "w") as f:
    json.dump(splits, f)
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump({"classes": CLASSES, "samples": expr.index.tolist(),
               "removed_pam50": to_drop, "n_genes": int(expr.shape[1]),
               "demo_mode": DEMO_MODE, "seed": SEED}, f)
print(f"Saved {len(splits)} outer splits")

## 5. Pipelines

Every step that looks at data (variance filter, scaling, PCA, hyperparameter search) sits inside a `Pipeline`,
so it is refitted on each training fold and never sees the test fold.

In [ ]:
class TopVarianceSelector(BaseEstimator, TransformerMixin):
    # Keep the k highest-variance genes, estimated on the training fold only.
    def __init__(self, k=5000):
        self.k = k
    def fit(self, X, y=None):
        v = np.asarray(X).var(axis=0)
        k = min(self.k, v.shape[0])
        self.idx_ = np.sort(np.argsort(v)[::-1][:k])
        return self
    def transform(self, X):
        return np.asarray(X)[:, self.idx_]

def tuned(estimator, grid):
    return GridSearchCV(estimator, grid, scoring="balanced_accuracy",
                        cv=StratifiedKFold(3, shuffle=True, random_state=SEED), n_jobs=-1)

def make_models(pca_dim=None):
    pre = [("var", TopVarianceSelector(TOP_VAR_GENES)), ("sc", StandardScaler())]
    if pca_dim is not None:
        pre.append(("pca", PCA(n_components=pca_dim, random_state=SEED)))
    return {
        "LogReg_L2": Pipeline(pre + [("clf", tuned(
            LogisticRegression(max_iter=5000, class_weight="balanced"),
            {"C": [0.01, 0.1, 1.0]}))]),
        "SVM_RBF": Pipeline(pre + [("clf", tuned(
            SVC(kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=SEED),
            {"C": [0.1, 1.0, 10.0]}))]),
        "RandomForest": Pipeline(pre + [("clf", RandomForestClassifier(
            n_estimators=500, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))]),
    }

def score(y_true, proba):
    pred = proba.argmax(axis=1)
    return {"bal_acc": balanced_accuracy_score(y_true, pred),
            "macro_f1": f1_score(y_true, pred, average="macro"),
            "macro_auc": roc_auc_score(y_true, proba, multi_class="ovr",
                                       average="macro", labels=np.arange(K))}

## 6. Classical baselines: genome-wide and compressed features

Genome-wide models show what a classical model can do with thousands of genes.
Compressed models (4, 6, 8 PCA components) show how much is lost by compression alone.
The gap between the two is the information the week 3 quantum student has to recover through distillation.

In [ ]:
rows = []
settings = [("genome-wide", None)] + [(f"PCA-{k}", k) for k in PCA_DIMS]
t0 = time.time()
for feat_name, k in settings:
    for s in splits:
        tr, te = np.array(s["train"]), np.array(s["test"])
        for name, model in make_models(k).items():
            m = clone(model).fit(X[tr], y[tr])
            r = score(y[te], m.predict_proba(X[te]))
            rows.append({"features": feat_name, "model": name,
                         "repeat": s["repeat"], "fold": s["fold"], **r})
    print(f"{feat_name:12s} done ({time.time() - t0:.0f}s elapsed)")

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "classical_results_per_fold.csv", index=False)

summary = (results.groupby(["features", "model"])[["bal_acc", "macro_f1", "macro_auc"]]
                  .agg(["mean", "std"]).round(3))
summary

In [ ]:
order = [f for f, _ in settings]
pivot = results.groupby(["features", "model"])["bal_acc"].agg(["mean", "std"]).reset_index()
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.25
models = pivot["model"].unique()
for j, mname in enumerate(models):
    sub = pivot[pivot["model"] == mname].set_index("features").loc[order]
    ax.bar(np.arange(len(order)) + (j - 1) * width, sub["mean"], width,
           yerr=sub["std"], capsize=3, label=mname)
ax.axhline(1 / K, ls="--", c="gray", lw=1, label="chance")
ax.set_xticks(np.arange(len(order)))
ax.set_xticklabels(order)
ax.set_ylabel("Balanced accuracy")
ax.set_title("Classical baselines, PAM50 genes excluded")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "classical_baselines.png", dpi=200)
plt.show()

## 7. Leakage check with shuffled labels

If the pipeline is leak-free, a model trained on shuffled labels should score close to chance (balanced accuracy near 1/K).
A clearly higher score means information is leaking from test to training data.

In [ ]:
rng = np.random.default_rng(SEED)
y_perm = rng.permutation(y)
perm_scores = []
for s in splits[:N_SPLITS]:
    tr, te = np.array(s["train"]), np.array(s["test"])
    m = clone(make_models(None)["LogReg_L2"]).fit(X[tr], y_perm[tr])
    perm_scores.append(score(y_perm[te], m.predict_proba(X[te]))["bal_acc"])
print(f"Shuffled-label balanced accuracy: {np.mean(perm_scores):.3f} +/- {np.std(perm_scores):.3f} "
      f"(chance = {1 / K:.3f})")

## 8. Save compressed features for the week 2 quantum kernels

For each split and each PCA size, the variance filter, scaler and PCA are fitted on the training fold,
then features are rescaled to [0, pi] (fitted on the training fold; test values are clipped),
which is a common input range for angle-encoding feature maps.

In [ ]:
for k in PCA_DIMS:
    store = {}
    for s in splits:
        tr, te = np.array(s["train"]), np.array(s["test"])
        prep = Pipeline([("var", TopVarianceSelector(TOP_VAR_GENES)), ("sc", StandardScaler()),
                         ("pca", PCA(n_components=k, random_state=SEED)),
                         ("mm", MinMaxScaler(feature_range=(0, np.pi)))])
        Z_tr = prep.fit_transform(X[tr])
        Z_te = np.clip(prep.transform(X[te]), 0, np.pi)
        key = f"r{s['repeat']}_f{s['fold']}"
        store[f"{key}_Xtr"], store[f"{key}_Xte"] = Z_tr.astype(np.float32), Z_te.astype(np.float32)
        store[f"{key}_ytr"], store[f"{key}_yte"] = y[tr], y[te]
        store[f"{key}_tr_idx"], store[f"{key}_te_idx"] = tr, te
    np.savez_compressed(OUT_DIR / f"compressed_pca{k}.npz", **store)
    print(f"Saved compressed_pca{k}.npz ({len(splits)} splits)")

## 9. Cross-fitted teacher soft labels for week 3

A teacher's predictions on its own training samples are overconfident, so they carry little extra information for the student.
Instead, for each outer training fold we produce **out-of-fold** probabilities with an inner cross-validation.
The teacher never sees the outer test fold. Its test-fold predictions are saved only for evaluation, never for training the student.

The cell also compares in-sample confidence with cross-fitted confidence, which shows why cross-fitting matters.

In [ ]:
teacher_store, conf_rows = {}, []
for s in splits:
    if s["repeat"] not in TEACHER_REPEATS:
        continue
    tr, te = np.array(s["train"]), np.array(s["test"])
    teacher = clone(make_models(None)[TEACHER_NAME])
    inner = StratifiedKFold(INNER_SPLITS, shuffle=True, random_state=SEED)
    soft_tr = cross_val_predict(teacher, X[tr], y[tr], cv=inner, method="predict_proba")
    teacher.fit(X[tr], y[tr])
    in_sample = teacher.predict_proba(X[tr])
    proba_te = teacher.predict_proba(X[te])

    key = f"r{s['repeat']}_f{s['fold']}"
    teacher_store[f"{key}_soft_tr"] = soft_tr.astype(np.float32)
    teacher_store[f"{key}_proba_te_eval_only"] = proba_te.astype(np.float32)
    teacher_store[f"{key}_tr_idx"], teacher_store[f"{key}_te_idx"] = tr, te
    conf_rows.append({"split": key,
                      "in_sample_mean_max_prob": in_sample.max(1).mean(),
                      "crossfit_mean_max_prob": soft_tr.max(1).mean(),
                      "crossfit_bal_acc_on_train": balanced_accuracy_score(y[tr], soft_tr.argmax(1)),
                      **{f"test_{m}": v for m, v in score(y[te], proba_te).items()}})
    print(f"{key}: teacher done")

np.savez_compressed(OUT_DIR / "teacher_soft_labels.npz", **teacher_store)
conf = pd.DataFrame(conf_rows)
conf.to_csv(OUT_DIR / "teacher_diagnostics.csv", index=False)
conf.round(3)

## 10. What to record before week 2

- The genome-wide versus PCA-k gap in balanced accuracy (section 6). This is the target for distillation.
- The shuffled-label score (section 7). It should be close to chance.
- In-sample versus cross-fitted teacher confidence (section 9). Cross-fitted confidence should be clearly lower.
- Which teacher to use. If random forest or SVM is clearly better than logistic regression in section 6, set `TEACHER_NAME` and re-run section 9.

Files in `outputs/week1/`: `splits.json`, `meta.json`, `classical_results_per_fold.csv`, `classical_baselines.png`,
`compressed_pca{4,6,8}.npz`, `teacher_soft_labels.npz`, `teacher_diagnostics.csv`.